# Rule-based Agent — Caro 11×11

**Tác giả:** Lê Tấn Phát  
**Phạm vi:** Giai đoạn 1 & 2 hoàn chỉnh — Luật thắng/chặn và nhận diện Pattern chiến thuật (Phần 1–6 hoàn chỉnh; Phần 7–10 sẽ hoàn thiện ở Giai đoạn 3)  
**Nguồn module:** `agents/rule_based_agent.py`  
**Lưu ý:** Mọi lần chạy nên bắt đầu bằng **Restart Kernel & Run All** để đảm bảo tính tái lập.


## Phần 1 — Câu hỏi nghiên cứu và phạm vi

**Câu hỏi trung tâm:**  
Việc bổ sung nhận diện pattern nâng cao (open-four, open-three, chuỗi có khoảng trống) có thực sự nâng cao chất lượng nước đi so với việc chỉ dùng luật thắng ngay/chặn thua kết hợp điểm vị trí cơ bản hay không?

**Phạm vi Phần 1–4 (Giai đoạn 1):**
- Xây dựng baseline chiến thuật: thắng ngay → chặn thua → chọn vị trí tốt.
- Kiểm chứng thực nghiệm cho thấy luật thắng/chặn hoạt động đúng ở cả 4 hướng và 2 phía.
- Chưa cài đặt pattern nâng cao (`use_patterns=False` ở Giai đoạn 1).

**Hạn chế đã biết ở Giai đoạn 1:**
- Nước dự phòng (fallback) chỉ dựa vào điểm vị trí đơn giản, chưa nhận diện open-four hay open-three.
- Không xử lý đe dọa kép (double threat) của đối thủ — đây là giới hạn tự nhiên của tầng luật hiện tại.
- Tập ứng viên fallback bán kính 2 có thể bỏ sót các vị trí chiến lược xa quân cờ hiện có.


## Phần 2 — Môi trường và cấu hình

In [1]:
from pathlib import Path
import os

# Chuyển về thư mục caro-11x11 để import module chung từ vị trí notebook.
if Path.cwd().name == 'notebooks' and (Path.cwd().parent / 'core').is_dir():
    os.chdir(Path.cwd().parent)
if not (Path.cwd() / 'core' / 'game_engine.py').is_file():
    raise RuntimeError('Hãy mở notebook từ thư mục caro-11x11 hoặc notebooks')

print(f'Thư mục làm việc: {Path.cwd()}')

Thư mục làm việc: D:\TTNTNC\AdvancedAI-Repo\caro-11x11


In [2]:
import numpy as np

from core import CaroGame
from core.game_utils import candidate_actions
from agents import RandomAgent

# Import trực tiếp từ module rule_based_agent (không chép lại thuật toán)
from agents.rule_based_agent import (
    RuleBasedAgent,
    is_winning_move,
    find_winning_actions,
    count_winning_threats,
    creates_open_four,
    creates_broken_four,
    creates_open_three,
    positional_score,
    DIRECTIONS,
)
from config.game_config import BOARD_SIZE, WIN_LENGTH, PLAYER_X, PLAYER_O, EMPTY

print('Import thành công.')

Import thành công.


In [3]:
# ---- Cấu hình thí nghiệm ----
CANDIDATE_RADIUS = 2       # Bán kính tập ứng viên cho fallback
TIEBREAK_SEED = None       # None = tất định; đặt số nguyên để ngẫu nhiên có kiểm soát
GAME_SEED = 42             # Seed cho RandomAgent trong thi đấu thử nghiệm
WIN_LENGTH_CFG = WIN_LENGTH  # 5 quân liên tiếp

# Khởi tạo agent baseline (Giai đoạn 1: use_patterns=False)
agent = RuleBasedAgent(
    name='RuleBased-Baseline',
    candidate_radius=CANDIDATE_RADIUS,
    use_patterns=False,
    seed=TIEBREAK_SEED,
)

print(f'Tác tử: {agent.name}')
print(f'Bán kính ứng viên: {agent.candidate_radius}')
print(f'use_patterns: {agent.use_patterns}')
print(f'Seed tie-breaking: {agent.seed}')
print(f'Kích thước bàn cờ: {BOARD_SIZE}×{BOARD_SIZE}')
print(f'Độ dài thắng: {WIN_LENGTH_CFG} quân liên tiếp')

Tác tử: RuleBased-Baseline
Bán kính ứng viên: 2
use_patterns: False
Seed tie-breaking: None
Kích thước bàn cờ: 11×11
Độ dài thắng: 5 quân liên tiếp


### Thuyết minh cấu hình và cách đo

**1. Công thức `positional_score(board, row, col, player)`**

Điểm gần tâm theo khoảng cách Chebyshev (với `center = BOARD_SIZE // 2 = 5`):

$$score_{tam} = center - \max(|r - center|,\ |c - center|)$$

Cộng thêm điểm thưởng lân cận (duyệt các ô trong cửa sổ bán kính 2 quanh `(row, col)`):

| Loại quân lân cận | Khoảng cách | Điểm cộng |
| :--- | :---: | :---: |
| Đồng minh (cùng màu `player`) | bán kính 1 | $+0.3$ |
| Đồng minh (cùng màu `player`) | bán kính 2 | $+0.1$ |
| Đối thủ | bán kính 2 (gồm cả bán kính 1) | $+0.05$ |

**2. Quy tắc phá hòa (tie-breaking)**

- `seed=None` (tất định): chọn nước có khóa lớn nhất theo `(positional_score, -manhattan_dist_to_center, -row, -col)` — tức điểm vị trí cao nhất, rồi khoảng cách Manhattan tới tâm nhỏ nhất, rồi tọa độ `(row, col)` nhỏ nhất.
- `seed != None`: dùng `np.random.default_rng(seed)` để chọn ngẫu nhiên có kiểm soát trong nhóm `top_group` gồm các nước có `positional_score` cao nhất bằng nhau.

**3. Phương pháp đếm `stats["nodes_visited"]`** (đặt lại về 0 ở đầu mỗi lượt đi)

- Tầng `win_now`: mỗi ô trống được kiểm tra tính là 1 node, cộng `+total_empty`.
- Tầng `block_urgent` (chỉ chạy nếu không có nước thắng ngay): quét thêm toàn bộ ô trống để tìm đe dọa của đối thủ, cộng `+total_empty`.
- Tầng `positional_fallback` (chỉ chạy nếu không có nước thắng và không phải chặn): cộng thêm số ứng viên trong bán kính `candidate_radius`, tức `+len(candidates)`.

## Phần 3 — Bộ vị trí chiến thuật

Dưới đây là các tình huống thử nghiệm được dựng thông qua chuỗi nước đi hợp lệ trên `CaroGame.step()` (đảm bảo cân bằng số quân theo đúng lượt đi: $|X| = |O|$ khi đến lượt X, $|X| = |O| + 1$ khi đến lượt O), kết hợp hàm tạo mảng trực tiếp cho các ca kiểm thử biên đặc biệt. Mỗi tình huống nêu rõ:
- Trạng thái bàn cờ
- Bên đến lượt
- Danh sách nước đi chấp nhận được và lý do

In [4]:
def make_board(x_stones, o_stones):
    """Dựng bàn cờ 11×11 bằng chuỗi nước đi hợp lệ qua CaroGame.step().

    X đi trước, hai bên xen kẽ. Số quân phải hợp lệ theo lượt đi:
      - len(x_stones) == len(o_stones)     -> đến lượt X
      - len(x_stones) == len(o_stones) + 1 -> đến lượt O
    Trả về bản sao của board (không giữ tham chiếu tới engine).
    """
    assert len(x_stones) - len(o_stones) in (0, 1), (
        f'Số quân không hợp lệ theo lượt đi: |X|={len(x_stones)}, |O|={len(o_stones)}'
    )
    game = CaroGame()
    for i, x_move in enumerate(x_stones):
        game.step(x_move)
        if i < len(o_stones):
            game.step(o_stones[i])
    return game.board.copy()

def show_board(board, title=''):
    """Hiển thị bàn cờ dạng text."""
    symbols = {0: '.', 1: 'X', -1: 'O'}
    if title:
        print(f'=== {title} ===')
    header = '   ' + ' '.join(f'{c:2}' for c in range(board.shape[1]))
    print(header)
    for r, row in enumerate(board):
        line = f'{r:2} ' + ' '.join(f'{symbols[int(v)]:2}' for v in row)
        print(line)
    print()

# Tình huống 1: X sắp thắng ngang (đến lượt X: |X| = |O| = 4)
board_win_h = make_board(
    x_stones=[(5, 1), (5, 2), (5, 3), (5, 4)],
    o_stones=[(0, 0), (0, 2), (0, 4), (0, 6)],
)
show_board(board_win_h, 'Tình huống 1: X có 4 quân ngang hàng 5')
print(f'Nước đi chấp nhận được (X): {find_winning_actions(board_win_h, PLAYER_X)}')
print('Lý do: Bất kỳ ô nào trong tập trên đều tạo chuỗi 5 quân → X thắng ngay.')
print()

=== Tình huống 1: X có 4 quân ngang hàng 5 ===
    0  1  2  3  4  5  6  7  8  9 10
 0 O  .  O  .  O  .  O  .  .  .  . 
 1 .  .  .  .  .  .  .  .  .  .  . 
 2 .  .  .  .  .  .  .  .  .  .  . 
 3 .  .  .  .  .  .  .  .  .  .  . 
 4 .  .  .  .  .  .  .  .  .  .  . 
 5 .  X  X  X  X  .  .  .  .  .  . 
 6 .  .  .  .  .  .  .  .  .  .  . 
 7 .  .  .  .  .  .  .  .  .  .  . 
 8 .  .  .  .  .  .  .  .  .  .  . 
 9 .  .  .  .  .  .  .  .  .  .  . 
10 .  .  .  .  .  .  .  .  .  .  . 

Nước đi chấp nhận được (X): [(5, 0), (5, 5)]
Lý do: Bất kỳ ô nào trong tập trên đều tạo chuỗi 5 quân → X thắng ngay.



In [5]:
# Tình huống 2: O đang đe dọa, X phải chặn (đến lượt X: |X| = |O| = 4)
board_block = make_board(
    x_stones=[(5, 5), (6, 6), (7, 7), (9, 1)],
    o_stones=[(0, 7), (1, 7), (2, 7), (3, 7)],
)
show_board(board_block, 'Tình huống 2: O có 4 quân dọc cột 7, X phải chặn')
o_threats = find_winning_actions(board_block, PLAYER_O)
print(f'Đe dọa của O (nước thắng ngay cho O): {o_threats}')
print(f'X không có nước thắng: {find_winning_actions(board_block, PLAYER_X)}')
print('Lý do: X phải chặn tại một trong các ô trên để O không thắng ngay.')
print()

=== Tình huống 2: O có 4 quân dọc cột 7, X phải chặn ===
    0  1  2  3  4  5  6  7  8  9 10
 0 .  .  .  .  .  .  .  O  .  .  . 
 1 .  .  .  .  .  .  .  O  .  .  . 
 2 .  .  .  .  .  .  .  O  .  .  . 
 3 .  .  .  .  .  .  .  O  .  .  . 
 4 .  .  .  .  .  .  .  .  .  .  . 
 5 .  .  .  .  .  X  .  .  .  .  . 
 6 .  .  .  .  .  .  X  .  .  .  . 
 7 .  .  .  .  .  .  .  X  .  .  . 
 8 .  .  .  .  .  .  .  .  .  .  . 
 9 .  X  .  .  .  .  .  .  .  .  . 
10 .  .  .  .  .  .  .  .  .  .  . 

Đe dọa của O (nước thắng ngay cho O): [(4, 7)]
X không có nước thắng: []
Lý do: X phải chặn tại một trong các ô trên để O không thắng ngay.



In [6]:
# Tình huống 3: Xung đột ưu tiên — X thắng ngay VÀ O đang đe dọa (đến lượt X: |X| = |O| = 4)
board_conflict = make_board(
    x_stones=[(5, 1), (5, 2), (5, 3), (5, 4)],
    o_stones=[(0, 9), (1, 9), (2, 9), (3, 9)],
)
show_board(board_conflict, 'Tình huống 3: X sắp thắng VÀ O đang đe dọa (xung đột ưu tiên)')
print(f'Nước thắng ngay của X: {find_winning_actions(board_conflict, PLAYER_X)}')
print(f'Đe dọa của O: {find_winning_actions(board_conflict, PLAYER_O)}')
print('Nước chấp nhận được: X phải chọn nước THẮNG NGAY (không được chặn O trước).')
print()

=== Tình huống 3: X sắp thắng VÀ O đang đe dọa (xung đột ưu tiên) ===
    0  1  2  3  4  5  6  7  8  9 10
 0 .  .  .  .  .  .  .  .  .  O  . 
 1 .  .  .  .  .  .  .  .  .  O  . 
 2 .  .  .  .  .  .  .  .  .  O  . 
 3 .  .  .  .  .  .  .  .  .  O  . 
 4 .  .  .  .  .  .  .  .  .  .  . 
 5 .  X  X  X  X  .  .  .  .  .  . 
 6 .  .  .  .  .  .  .  .  .  .  . 
 7 .  .  .  .  .  .  .  .  .  .  . 
 8 .  .  .  .  .  .  .  .  .  .  . 
 9 .  .  .  .  .  .  .  .  .  .  . 
10 .  .  .  .  .  .  .  .  .  .  . 

Nước thắng ngay của X: [(5, 0), (5, 5)]
Đe dọa của O: [(4, 9)]
Nước chấp nhận được: X phải chọn nước THẮNG NGAY (không được chặn O trước).



In [7]:
# Tình huống 4: Bàn cờ trống — fallback vị trí (X đi nước đầu: |X| = |O| = 0)
board_empty = make_board([], [])
show_board(board_empty, 'Tình huống 4: Bàn cờ trống — không có đe dọa trực tiếp')
print('Nước đi chấp nhận được: (5,5) — tâm bàn cờ (điểm vị trí cao nhất).')
print('Lý do: Fallback positional_score ưu tiên tâm bàn cờ theo khoảng cách Chebyshev.')
print()

=== Tình huống 4: Bàn cờ trống — không có đe dọa trực tiếp ===
    0  1  2  3  4  5  6  7  8  9 10
 0 .  .  .  .  .  .  .  .  .  .  . 
 1 .  .  .  .  .  .  .  .  .  .  . 
 2 .  .  .  .  .  .  .  .  .  .  . 
 3 .  .  .  .  .  .  .  .  .  .  . 
 4 .  .  .  .  .  .  .  .  .  .  . 
 5 .  .  .  .  .  .  .  .  .  .  . 
 6 .  .  .  .  .  .  .  .  .  .  . 
 7 .  .  .  .  .  .  .  .  .  .  . 
 8 .  .  .  .  .  .  .  .  .  .  . 
 9 .  .  .  .  .  .  .  .  .  .  . 
10 .  .  .  .  .  .  .  .  .  .  . 

Nước đi chấp nhận được: (5,5) — tâm bàn cờ (điểm vị trí cao nhất).
Lý do: Fallback positional_score ưu tiên tâm bàn cờ theo khoảng cách Chebyshev.



## Phần 4 — Kiểm chứng luật thắng và chặn

Trong phần này, ta quan sát trực tiếp phản ứng của `RuleBasedAgent` trên các bộ vị trí chiến thuật ở:
- Cả 4 hướng: ngang, dọc, chéo chính, chéo phụ
- Cả 2 phía: X và O
- Vị trí sát biên bàn cờ
- Chuỗi dài hơn 5 quân
- Đồng thời có cơ hội thắng lẫn đe dọa thua

In [8]:
def test_scenario(board, player, expected_rule, acceptable_actions=None, title=''):
    """Chạy agent trên một tình huống và in kết quả quan sát."""
    _agent = RuleBasedAgent(seed=None)  # tất định
    board_copy = board.copy()
    action = _agent.choose_action(board_copy, player)
    info = _agent.explain_last_action()
    
    # Kiểm tra bất biến board
    board_unchanged = np.array_equal(board, board_copy)
    rule_ok = info['rule_triggered'] == expected_rule
    action_ok = (acceptable_actions is None) or (action in acceptable_actions)
    
    status = '✓ PASS' if (rule_ok and action_ok and board_unchanged) else '✗ FAIL'
    player_sym = 'X' if player == PLAYER_X else 'O'
    
    print(f'{status} [{title}]')
    print(f'  Bên đến lượt: {player_sym}')
    print(f'  Luật kích hoạt: {info["rule_triggered"]} (kỳ vọng: {expected_rule})')
    print(f'  Nước chọn: {action}')
    print(f'  Lý do: {info["reason"]}')
    print(f'  Board bất biến: {board_unchanged}')
    if acceptable_actions:
        print(f'  Nước đi chấp nhận được: {acceptable_actions}')
        print(f'  Nước chọn hợp lệ: {action_ok}')
    print()
    return status

print('=== PHẦN 4A: Kiểm chứng thắng ngay (win_now) — 4 hướng × 2 phía ===')
print()

# Ngang — X thắng (lượt X: 4 và 4)
b = make_board([(5, 1), (5, 2), (5, 3), (5, 4)], [(0, 0), (0, 2), (0, 4), (0, 6)])
test_scenario(b, PLAYER_X, 'win_now', find_winning_actions(b, PLAYER_X),
              'Ngang — X thắng')

# Ngang — O thắng (lượt O: 5 và 4)
b = make_board([(0, 0), (0, 2), (0, 4), (2, 0), (2, 2)], [(7, 1), (7, 2), (7, 3), (7, 4)])
test_scenario(b, PLAYER_O, 'win_now', find_winning_actions(b, PLAYER_O),
              'Ngang — O thắng')

# Dọc — X thắng (lượt X: 4 và 4)
b = make_board([(0, 5), (1, 5), (2, 5), (3, 5)], [(0, 0), (0, 2), (0, 4), (0, 6)])
test_scenario(b, PLAYER_X, 'win_now', find_winning_actions(b, PLAYER_X),
              'Dọc — X thắng')

# Dọc — O thắng (lượt O: 5 và 4)
b = make_board([(0, 0), (0, 2), (0, 4), (2, 0), (2, 2)], [(1, 3), (2, 3), (3, 3), (4, 3)])
test_scenario(b, PLAYER_O, 'win_now', find_winning_actions(b, PLAYER_O),
              'Dọc — O thắng')

# Chéo chính — X thắng (lượt X: 4 và 4)
b = make_board([(0, 0), (1, 1), (2, 2), (3, 3)], [(0, 2), (0, 4), (2, 0), (4, 0)])
test_scenario(b, PLAYER_X, 'win_now', find_winning_actions(b, PLAYER_X),
              'Chéo chính — X thắng')

# Chéo phụ — O thắng (lượt O: 5 và 4)
b = make_board([(0, 0), (0, 2), (0, 4), (2, 0), (2, 2)], [(1, 9), (2, 8), (3, 7), (4, 6)])
test_scenario(b, PLAYER_O, 'win_now', find_winning_actions(b, PLAYER_O),
              'Chéo phụ — O thắng')

=== PHẦN 4A: Kiểm chứng thắng ngay (win_now) — 4 hướng × 2 phía ===

✓ PASS [Ngang — X thắng]
  Bên đến lượt: X
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (5, 5)
  Lý do: Thắng ngay tại (5, 5) — tìm thấy 2 nước thắng, chọn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(5, 0), (5, 5)]
  Nước chọn hợp lệ: True

✓ PASS [Ngang — O thắng]
  Bên đến lượt: O
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (7, 5)
  Lý do: Thắng ngay tại (7, 5) — tìm thấy 2 nước thắng, chọn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(7, 0), (7, 5)]
  Nước chọn hợp lệ: True

✓ PASS [Dọc — X thắng]
  Bên đến lượt: X
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (4, 5)
  Lý do: Thắng ngay tại (4, 5) — tìm thấy 1 nước thắng, chọn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(4, 5)]
  Nước chọn hợp lệ: True

✓ PASS [Dọc — O thắng]
  Bên đến lượt: O
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (5

'✓ PASS'

In [9]:
print('=== PHẦN 4B: Kiểm chứng chặn thua (block_urgent) — 4 hướng + biên ===')
print()

# Ngang — X chặn O (lượt X: 4 và 4)
b = make_board([(0, 0), (0, 2), (2, 0), (2, 2)], [(7, 3), (7, 4), (7, 5), (7, 6)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Ngang — X chặn O')

# Dọc — X chặn O
b = make_board([(5, 0), (5, 2), (7, 0), (7, 2)], [(0, 7), (1, 7), (2, 7), (3, 7)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Dọc — X chặn O')

# Chéo chính — X chặn O
b = make_board([(0, 2), (0, 4), (2, 0), (8, 8)], [(1, 1), (2, 2), (3, 3), (4, 4)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Chéo chính — X chặn O')

# Chéo phụ — X chặn O
b = make_board([(0, 0), (0, 2), (2, 0), (8, 8)], [(1, 9), (2, 8), (3, 7), (4, 6)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Chéo phụ — X chặn O')

# Biên dưới — O sát hàng 10
b = make_board([(5, 0), (5, 2), (7, 0), (7, 2)], [(10, 0), (10, 1), (10, 2), (10, 3)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Sát biên dưới — X chặn O')

# Biên trái — O sát cột 0
b = make_board([(5, 5), (5, 7), (7, 5), (7, 7)], [(0, 0), (1, 0), (2, 0), (3, 0)])
test_scenario(b, PLAYER_X, 'block_urgent', find_winning_actions(b, PLAYER_O),
              'Sát biên trái — X chặn O')

=== PHẦN 4B: Kiểm chứng chặn thua (block_urgent) — 4 hướng + biên ===

✓ PASS [Ngang — X chặn O]
  Bên đến lượt: X
  Luật kích hoạt: block_urgent (kỳ vọng: block_urgent)
  Nước chọn: (7, 7)
  Lý do: Chặn thua tại (7, 7) — đối thủ có 2 đe dọa thắng ngay, chặn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(7, 2), (7, 7)]
  Nước chọn hợp lệ: True

✓ PASS [Dọc — X chặn O]
  Bên đến lượt: X
  Luật kích hoạt: block_urgent (kỳ vọng: block_urgent)
  Nước chọn: (4, 7)
  Lý do: Chặn thua tại (4, 7) — đối thủ có 1 đe dọa thắng ngay, chặn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(4, 7)]
  Nước chọn hợp lệ: True

✓ PASS [Chéo chính — X chặn O]
  Bên đến lượt: X
  Luật kích hoạt: block_urgent (kỳ vọng: block_urgent)
  Nước chọn: (5, 5)
  Lý do: Chặn thua tại (5, 5) — đối thủ có 2 đe dọa thắng ngay, chặn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(0, 0), (5, 5)]
  Nước chọn hợp lệ: True

✓ PASS [Chéo phụ — X chặn O]
  Bên đế

'✓ PASS'

In [10]:
print('=== PHẦN 4C: Chuỗi đặc biệt — >5 quân và bị chặn 2 đầu ===')
print()

# Chuỗi >5 quân: X tại (0,1)(0,2)(0,3)(0,5)(0,6), ô (0,4) trống → nối 6 (lượt X: 5 và 5)
b = make_board([(0, 1), (0, 2), (0, 3), (0, 5), (0, 6)],
               [(2, 0), (2, 2), (4, 0), (4, 2), (6, 0)])
print(f'is_winning_move(board, 0, 4, X) = {is_winning_move(b, 0, 4, PLAYER_X)}')
test_scenario(b, PLAYER_X, 'win_now', [(0, 4)], 'Chuỗi 6 quân — X nối ở giữa')

# Chuỗi bị chặn 2 đầu: O X X . X X O, đặt X vào giữa → 5 quân liên tiếp (lượt X: 4 và 4)
b = make_board([(0, 2), (0, 3), (0, 5), (0, 6)], [(0, 1), (0, 7), (2, 0), (2, 2)])
print(f'is_winning_move(board, 0, 4, X) = {is_winning_move(b, 0, 4, PLAYER_X)}')
test_scenario(b, PLAYER_X, 'win_now', [(0, 4)], 'Chuỗi 5 bị chặn 2 đầu — X điền vào giữa')

=== PHẦN 4C: Chuỗi đặc biệt — >5 quân và bị chặn 2 đầu ===

is_winning_move(board, 0, 4, X) = True
✓ PASS [Chuỗi 6 quân — X nối ở giữa]
  Bên đến lượt: X
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (0, 4)
  Lý do: Thắng ngay tại (0, 4) — tìm thấy 1 nước thắng, chọn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(0, 4)]
  Nước chọn hợp lệ: True

is_winning_move(board, 0, 4, X) = True
✓ PASS [Chuỗi 5 bị chặn 2 đầu — X điền vào giữa]
  Bên đến lượt: X
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (0, 4)
  Lý do: Thắng ngay tại (0, 4) — tìm thấy 1 nước thắng, chọn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(0, 4)]
  Nước chọn hợp lệ: True



'✓ PASS'

In [11]:
print('=== PHẦN 4D: Xung đột ưu tiên — thắng ngay > chặn thua ===')
print()

# X có 4 quân ngang VÀ O có 4 quân dọc → X phải chọn thắng ngay (lượt X: 4 và 4)
b = make_board(
    [(5, 1), (5, 2), (5, 3), (5, 4)],
    [(0, 9), (1, 9), (2, 9), (3, 9)],
)
x_wins = find_winning_actions(b, PLAYER_X)
o_threats = find_winning_actions(b, PLAYER_O)
print(f'Nước thắng ngay của X: {x_wins}')
print(f'Đe dọa của O: {o_threats}')
test_scenario(b, PLAYER_X, 'win_now', x_wins, 'Xung đột: X thắng > chặn O')

# O có 4 quân dọc VÀ X có 4 quân ngang → O phải chọn thắng ngay (lượt O: 5 và 4)
b = make_board(
    [(8, 2), (8, 3), (8, 4), (8, 5), (10, 10)],
    [(0, 3), (1, 3), (2, 3), (3, 3)],
)
o_wins = find_winning_actions(b, PLAYER_O)
x_threats = find_winning_actions(b, PLAYER_X)
print(f'Nước thắng ngay của O: {o_wins}')
print(f'Đe dọa của X: {x_threats}')
test_scenario(b, PLAYER_O, 'win_now', o_wins, 'Xung đột: O thắng > chặn X')

=== PHẦN 4D: Xung đột ưu tiên — thắng ngay > chặn thua ===

Nước thắng ngay của X: [(5, 0), (5, 5)]
Đe dọa của O: [(4, 9)]
✓ PASS [Xung đột: X thắng > chặn O]
  Bên đến lượt: X
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (5, 5)
  Lý do: Thắng ngay tại (5, 5) — tìm thấy 2 nước thắng, chọn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(5, 0), (5, 5)]
  Nước chọn hợp lệ: True

Nước thắng ngay của O: [(4, 3)]
Đe dọa của X: [(8, 1), (8, 6)]
✓ PASS [Xung đột: O thắng > chặn X]
  Bên đến lượt: O
  Luật kích hoạt: win_now (kỳ vọng: win_now)
  Nước chọn: (4, 3)
  Lý do: Thắng ngay tại (4, 3) — tìm thấy 1 nước thắng, chọn theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(4, 3)]
  Nước chọn hợp lệ: True



'✓ PASS'

In [12]:
print('=== PHẦN 4E: Trường hợp biên — bàn trống và bàn gần đầy ===')
print()

# Bàn cờ trống (dựng qua CaroGame, X đi nước đầu)
b_empty = make_board([], [])
test_scenario(b_empty, PLAYER_X, 'positional_fallback', [(5, 5)], 'Bàn trống — X đi nước đầu')

# Bàn cờ chỉ còn 1 ô trống.
# GHI CHÚ: đây là trạng thái biên GIẢ ĐỊNH, dựng trực tiếp bằng numpy (toàn bộ ô là X
# trừ (10, 10)); không phải thế cờ hợp lệ theo luật engine, chỉ để kiểm tra hợp đồng
# "luôn trả về ô trống duy nhất".
b_one = np.ones((11, 11), dtype=np.int8)
b_one[10, 10] = EMPTY
print('Ô trống duy nhất: (10, 10)')
_agent_one = RuleBasedAgent()
action_one = _agent_one.choose_action(b_one, PLAYER_X)
print(f'Nước chọn: {action_one} (kỳ vọng: (10, 10))')
print(f'Kết quả: {"✓ PASS" if action_one == (10, 10) else "✗ FAIL"}')

=== PHẦN 4E: Trường hợp biên — bàn trống và bàn gần đầy ===



✓ PASS [Bàn trống — X đi nước đầu]
  Bên đến lượt: X
  Luật kích hoạt: positional_fallback (kỳ vọng: positional_fallback)
  Nước chọn: (5, 5)
  Lý do: Dự phòng vị trí tại (5, 5) — không có đe dọa trực tiếp, chọn từ 1 ứng viên bán kính 2 theo tie-breaking.
  Board bất biến: True
  Nước đi chấp nhận được: [(5, 5)]
  Nước chọn hợp lệ: True

Ô trống duy nhất: (10, 10)
Nước chọn: (10, 10) (kỳ vọng: (10, 10))
Kết quả: ✓ PASS


In [13]:
print('=== PHẦN 4F: Thống kê nodes_visited và thinking_time ===')
print()

_agent_stats = RuleBasedAgent()
# Lượt X: |X| = |O| = 4
b_test = make_board([(5, 1), (5, 2), (5, 3), (5, 4)], [(0, 0), (0, 2), (0, 4), (0, 6)])

action = _agent_stats.choose_action(b_test, PLAYER_X)
print(f'Nước chọn: {action}')
print(f'nodes_visited: {_agent_stats.stats["nodes_visited"]}')
print(f'thinking_time: {_agent_stats.stats["thinking_time"]:.6f} giây')
print(f'Thông tin quyết định: {_agent_stats.explain_last_action()}')

=== PHẦN 4F: Thống kê nodes_visited và thinking_time ===

Nước chọn: (5, 5)
nodes_visited: 113
thinking_time: 0.000557 giây
Thông tin quyết định: {'rule_triggered': 'win_now', 'selected_action': (5, 5), 'candidate_count': 2, 'matching_actions': [(5, 0), (5, 5)], 'reason': 'Thắng ngay tại (5, 5) — tìm thấy 2 nước thắng, chọn theo tie-breaking.'}


### Nhận xét và phân tích kết quả Phần 4

Các nhận xét dưới đây dựa trên output thực tế của lần chạy ngay phía trên (18 ca `✓ PASS`, 0 ca `✗ FAIL`, không cell nào báo lỗi).

**1. Tính đúng đắn của luật thắng ngay và chặn thua**
- Toàn bộ các tình huống `win_now` (Phần 4A) ở cả 4 hướng (ngang, dọc, chéo chính, chéo phụ) và cả 2 phía X/O đều kích hoạt đúng luật kỳ vọng, nước chọn nằm trong tập `find_winning_actions`.
- Toàn bộ các tình huống `block_urgent` (Phần 4B) ở 4 hướng, kể cả sát biên dưới và sát biên trái, đều kích hoạt đúng luật và nước chọn nằm trong tập đe dọa của đối thủ.
- Các chuỗi đặc biệt (Phần 4C) được nhận diện đúng: nối thành chuỗi 6 quân tại `(0, 4)` và chuỗi 5 quân bị chặn cả 2 đầu vẫn được tính là thắng (`is_winning_move = True`), phù hợp luật của engine.
- Ở mọi ca, cờ `Board bất biến: True` — `get_action` không sửa mảng `board` đầu vào.

**2. Ưu tiên tuyệt đối của `win_now` so với `block_urgent`**
- Ở Phần 4D, X có nước thắng `[(5, 0), (5, 5)]` trong khi O đồng thời đe dọa tại `(4, 9)`: agent vẫn chọn `win_now` tại `(5, 5)` thay vì chặn O. Tình huống đảo bên (O thắng tại `(4, 3)` dù X đe dọa `(8, 1)`, `(8, 6)`) cho kết quả đối xứng.
- Lý do cấu trúc: `get_action` chỉ xuống tầng `block_urgent` khi tầng `win_now` không tìm được nước thắng nào, nên thắng ngay luôn được xét trước chặn thua. Điều này đúng về mặt lý thuyết vì thắng ngay kết thúc ván trước khi đối thủ kịp đi.

**3. Giới hạn quan sát được ở `block_urgent`**
- Khi đối thủ có chuỗi 4 quân hở cả hai đầu (ví dụ ca ngang `[(7, 2), (7, 7)]`, chéo chính `[(0, 0), (5, 5)]`, chéo phụ `[(0, 10), (5, 5)]`), agent chỉ chặn được một đầu; đối thủ vẫn thắng ở đầu còn lại ở lượt sau. Đây là giới hạn tự nhiên của luật 1 nước (đe dọa kép), không phải lỗi cài đặt, và là lý do cần nhận diện pattern sớm hơn (open-three) ở Giai đoạn 2.
- Các ca chỉ có một đầu bị đe dọa (dọc tại `(4, 7)`, sát biên dưới `(10, 4)`, sát biên trái `(4, 0)`) được chặn trọn vẹn.

**4. Chỉ số `nodes_visited` và `thinking_time` (Phần 4F)**
- Với thế X có 4 quân ngang, agent kích hoạt `win_now` ngay ở tầng 1 nên `nodes_visited = 113`, đúng bằng số ô trống (`121 - 8 = 113`) theo phương pháp đếm ở Phần 2; tầng `block_urgent` và `positional_fallback` không chạy nên không cộng thêm.
- `thinking_time` cỡ nửa mili-giây (khoảng 0.0005 giây), thấp hơn rất nhiều so với mốc hợp tác 1.9 giây; chi phí chủ yếu là một lượt quét toàn bàn (121 ô) với phép đếm 4 hướng cho mỗi ô trống.
- Lưu ý: giá trị `thinking_time` phụ thuộc máy và thay đổi nhẹ giữa các lần chạy; `nodes_visited` thì tất định.

---

## Phần 5 — Định nghĩa pattern và độ ưu tiên

### 5.1. Định nghĩa toán học và chiến thuật của các Pattern

Thay vì liệt kê cứng các mẫu chuỗi ký tự dễ bỏ sót ở biên hoặc khoảng trống nhảy, `agents/rule_based_agent.py` định nghĩa các pattern dựa trên **số mối đe dọa thắng ngay phát sinh** sau khi giả lập đặt quân của `player` vào ô trống `(row, col)` (trên bản sao cục bộ `board.copy()`, đảm bảo mảng `board` gốc hoàn toàn bất biến):

1. **`count_winning_threats(board, row, col, player, win_length=5) -> int`:**
   - Nếu ô `(row, col)` không trống hoặc bản thân `(row, col)` đã là nước thắng ngay (`is_winning_move == True`), trả về `0`.
   - Giả lập đặt `player` vào `(row, col)`, sau đó quét dọc theo 4 trục hướng đi qua `(row, col)` trong bán kính `1 .. win_length - 1` ở cả hai chiều. Đếm số ô trống `(nr, nc)` **mới trở thành nước thắng ngay** (`is_winning_move(board_after, nr, nc, player) == True` và `is_winning_move(board, nr, nc, player) == False`).

2. **`open_four` — Tứ mở / Đa đe dọa thắng (`creates_open_four`):**
   - **Điều kiện:** `count_winning_threats(board, row, col, player) >= 2`.
   - **Hình thái điển hình:** Chuỗi 4 quân liên tiếp mở cả hai đầu `. X X X X .` (tạo 2 điểm thắng độc lập ở hai đầu) hoặc nước giao điểm tạo đồng thời 2 đường 4 quân có điểm thắng khác nhau. Khi đã tạo được `open_four`, đối thủ chỉ có thể chặn 1 điểm nên `player` chắc chắn thắng ở lượt kế tiếp.

3. **`broken_four` — Tứ gãy / Tứ nửa mở (`creates_broken_four`):**
   - **Điều kiện:** `count_winning_threats(board, row, col, player) == 1`.
   - **Hình thái điển hình:**
     + Chuỗi 4 quân bị chặn 1 đầu bởi quân đối phương (`O X X X X .`) hoặc sát biên bàn cờ (`| X X X X .`).
     + Chuỗi 4 quân có khoảng trống ở giữa (`X X . X X` hoặc `X . X X X`), chỉ có duy nhất 1 ô trống ở khe giữa để hoàn tất 5 quân liên tiếp.
   - **Ý nghĩa chiến thuật:** Buộc đối thủ phải dùng lượt kế tiếp để chặn (`block_urgent`), giúp bên tấn công giành quyền chủ động (forcing move).

4. **`open_three` — Tam mở liền & Tam mở nhảy (`creates_open_three`):**
   - **Điều kiện:** Ô `(row, col)` chưa thắng ngay, `count_winning_threats == 0`, nhưng sau khi đặt `player` vào `(row, col)`, trên 4 trục đi qua `(row, col)` xuất hiện ít nhất 1 ô trống `(nr, nc)` mới thỏa mãn `creates_open_four(board_after, nr, nc, player) == True`.
   - **Hình thái điển hình:**
     + Tam mở liền: `. . X X X . .` (bước kế tiếp đánh vào 1 trong 2 đầu sát chuỗi sẽ tạo `. X X X X .`).
     + Tam mở nhảy (có khoảng trống): `. X . X X .` hoặc `. X X . X .` (bước kế tiếp lấp vào khe trống ở giữa sẽ tạo `. X X X X .`).
   - **Loại trừ nhận diện nhầm (False Positive):** Nếu chuỗi 3 bị chặn bởi quân đối phương hoặc biên sao cho không thể mở rộng thành `open_four` (ví dụ `O X X X . .` bị chặn sát đầu trái, đánh tiếp chỉ tạo `O X X X X .` tức `broken_four`), hàm trả về `False`.

---

### 5.2. Bảng phân cấp 8 tầng ưu tiên (`use_patterns=True`)

| Thứ tự | Tên luật (`rule_triggered`) | Điều kiện kích hoạt | Phạm vi quét | Lý do xếp thứ tự ưu tiên |
| :---: | :--- | :--- | :--- | :--- |
| **1** | `win_now` | `is_winning_move(board, r, c, player)` | Toàn bàn (`total_empty`) | Thắng ngay lập tức kết thúc ván cờ ở lượt hiện tại. |
| **2** | `block_urgent` | `is_winning_move(board, r, c, opponent)` | Toàn bàn (`total_empty`) | Nếu không chặn, đối thủ sẽ thắng ngay ở lượt kế tiếp. |
| **3** | `attack_open_four` | `creates_open_four(board, r, c, player)` | Ứng viên `radius=2` | Tạo $\ge 2$ mối đe dọa thắng ngay; đối thủ không thể chặn hết ở lượt sau. |
| **4** | `block_open_four` | `creates_open_four(board, r, c, opponent)` | Ứng viên `radius=2` | Ngăn đối thủ tạo `open_four` (nếu để đối thủ tạo xong sẽ không thể cứu vãn). |
| **5** | `attack_broken_four` | `creates_broken_four(board, r, c, player)` | Ứng viên `radius=2` | Tạo 1 đe dọa thắng trực tiếp buộc đối thủ phải đỡ, chiếm tiên thủ trước khi xét tam mở. |
| **6** | `attack_open_three` | `creates_open_three(board, r, c, player)` | Ứng viên `radius=2` | Tạo bàn đạp lên `open_four` ở lượt kế tiếp; ưu tiên tấn công trước khi chặn tam mở. |
| **7** | `block_open_three` | `creates_open_three(board, r, c, opponent)` | Ứng viên `radius=2` | Chặn sớm bàn đạp `open_three` của đối thủ trước khi biến thành đe dọa cấp 4. |
| **8** | `positional_fallback` | Chấm điểm `positional_score(board, r, c, player)` | Ứng viên `radius=2` | Khi chưa có mẫu hình chiến thuật nào, ưu tiên ô gần tâm và liên kết quân. |

**Cơ chế đếm `stats["nodes_visited"]` khi `use_patterns=True`:**
- Tầng 1 (`win_now`) và Tầng 2 (`block_urgent`): mỗi tầng duyệt qua toàn bộ ô trống cộng `+total_empty`.
- Tầng 3 đến Tầng 8: sinh tập `candidates = candidate_actions(board, radius=self.candidate_radius)` một lần; mỗi tầng được kiểm tra sẽ cộng thêm `+len(candidates)` vào `nodes_visited` cho đến khi có tầng khớp.

In [14]:
print('=== PHẦN 5A: Thực nghiệm đối chứng Open-Four và Broken-Four (Positive vs. Negative) ===')
print()

# 1. Open-Four chuẩn: X có (5,3),(5,4),(5,5), hai đầu (5,2) và (5,6) đều trống và có không gian
b_open4 = make_board(
    x_stones=[(5, 3), (5, 4), (5, 5)],
    o_stones=[(0, 0), (0, 2), (0, 4)],
)
show_board(b_open4, '5A.1 — X có 3 quân mở 2 đầu tại hàng 5: xét nước (5, 6)')
threats_5_6 = count_winning_threats(b_open4, 5, 6, PLAYER_X)
is_o4 = creates_open_four(b_open4, 5, 6, PLAYER_X)
is_b4 = creates_broken_four(b_open4, 5, 6, PLAYER_X)
print(f'  Nước (5, 6): count_winning_threats={threats_5_6}, creates_open_four={is_o4}, creates_broken_four={is_b4}')
assert threats_5_6 == 2 and is_o4 is True and is_b4 is False
print('  -> ✓ PASS: Chuỗi . X X X X . tạo đúng 2 mối đe dọa thắng tại (5, 2) và (5, 7).\n')

# 2. Đối chứng Negative / Broken-Four: Bị quân O chặn 1 đầu tại (5, 2)
b_blocked4 = make_board(
    x_stones=[(5, 3), (5, 4), (5, 5)],
    o_stones=[(5, 2), (0, 2), (0, 4)],
)
threats_blk = count_winning_threats(b_blocked4, 5, 6, PLAYER_X)
o4_blk = creates_open_four(b_blocked4, 5, 6, PLAYER_X)
b4_blk = creates_broken_four(b_blocked4, 5, 6, PLAYER_X)
print(f'[5A.2 — Bị O chặn tại (5,2), X đánh (5,6)]: threats={threats_blk}, open_four={o4_blk}, broken_four={b4_blk}')
assert threats_blk == 1 and o4_blk is False and b4_blk is True
print('  -> ✓ PASS: Không nhận diện nhầm là open_four; phân loại chính xác là broken_four (1 đầu thắng tại (5,7)).\n')

# 3. Đối chứng Sát biên: X có (5,0),(5,1),(5,2), đánh (5,3) -> đầu trái chạm biên cột 0
b_edge4 = make_board(
    x_stones=[(5, 0), (5, 1), (5, 2)],
    o_stones=[(0, 0), (0, 2), (0, 4)],
)
threats_edge = count_winning_threats(b_edge4, 5, 3, PLAYER_X)
o4_edge = creates_open_four(b_edge4, 5, 3, PLAYER_X)
b4_edge = creates_broken_four(b_edge4, 5, 3, PLAYER_X)
print(f'[5A.3 — Sát biên trái | X X X ., X đánh (5,3)]: threats={threats_edge}, open_four={o4_edge}, broken_four={b4_edge}')
assert threats_edge == 1 and o4_edge is False and b4_edge is True
print('  -> ✓ PASS: Sát biên chỉ còn 1 điểm thắng tại (5,4) nên là broken_four.\n')

# 4. Broken-Four dạng có khoảng trống giữa (X X . X -> đánh thêm X tạo X X . X X)
b_gap4 = make_board(
    x_stones=[(5, 2), (5, 3), (5, 5)],
    o_stones=[(0, 0), (0, 2), (0, 4)],
)
threats_gap = count_winning_threats(b_gap4, 5, 6, PLAYER_X)
o4_gap = creates_open_four(b_gap4, 5, 6, PLAYER_X)
b4_gap = creates_broken_four(b_gap4, 5, 6, PLAYER_X)
print(f'[5A.4 — Chuỗi đứt quãng X X . X, X đánh (5,6) tạo X X . X X]: threats={threats_gap}, open_four={o4_gap}, broken_four={b4_gap}')
assert threats_gap == 1 and o4_gap is False and b4_gap is True
print('  -> ✓ PASS: Chỉ ô giữa (5,4) mới nối thành 5 quân nên là broken_four.\n')

# 5. Đối chứng False Positive: Quân đối phương O nằm chắn giữa (X X O X -> đánh (5,6) thành X X O X X)
b_false4 = make_board(
    x_stones=[(5, 2), (5, 3), (5, 5)],
    o_stones=[(5, 4), (0, 2), (0, 4)],
)
threats_false = count_winning_threats(b_false4, 5, 6, PLAYER_X)
o4_false = creates_open_four(b_false4, 5, 6, PLAYER_X)
b4_false = creates_broken_four(b_false4, 5, 6, PLAYER_X)
print(f'[5A.5 — Bị O ngắt giữa X X O X, X đánh (5,6) tạo X X O X X]: threats={threats_false}, open_four={o4_false}, broken_four={b4_false}')
assert threats_false == 0 and o4_false is False and b4_false is False
print('  -> ✓ PASS: Cả hai hàm đều trả về False vì O đã chặn khe (5,4).')

=== PHẦN 5A: Thực nghiệm đối chứng Open-Four và Broken-Four (Positive vs. Negative) ===

=== 5A.1 — X có 3 quân mở 2 đầu tại hàng 5: xét nước (5, 6) ===
    0  1  2  3  4  5  6  7  8  9 10
 0 O  .  O  .  O  .  .  .  .  .  . 
 1 .  .  .  .  .  .  .  .  .  .  . 
 2 .  .  .  .  .  .  .  .  .  .  . 
 3 .  .  .  .  .  .  .  .  .  .  . 
 4 .  .  .  .  .  .  .  .  .  .  . 
 5 .  .  .  X  X  X  .  .  .  .  . 
 6 .  .  .  .  .  .  .  .  .  .  . 
 7 .  .  .  .  .  .  .  .  .  .  . 
 8 .  .  .  .  .  .  .  .  .  .  . 
 9 .  .  .  .  .  .  .  .  .  .  . 
10 .  .  .  .  .  .  .  .  .  .  . 

  Nước (5, 6): count_winning_threats=2, creates_open_four=True, creates_broken_four=False
  -> ✓ PASS: Chuỗi . X X X X . tạo đúng 2 mối đe dọa thắng tại (5, 2) và (5, 7).

[5A.2 — Bị O chặn tại (5,2), X đánh (5,6)]: threats=1, open_four=False, broken_four=True
  -> ✓ PASS: Không nhận diện nhầm là open_four; phân loại chính xác là broken_four (1 đầu thắng tại (5,7)).

[5A.3 — Sát biên trái | X X X ., X đánh (5,3

In [15]:
print('=== PHẦN 5B: Thực nghiệm đối chứng Open-Three (Tam mở liền, Tam mở nhảy vs. Tam bị chặn) ===')
print()

# 1. Tam mở liền: X có (5,4),(5,5), đánh (5,6) tạo . . X X X . .
b_open3_contig = make_board(
    x_stones=[(5, 4), (5, 5)],
    o_stones=[(0, 0), (0, 2)],
)
o3_contig = creates_open_three(b_open3_contig, 5, 6, PLAYER_X)
print(f'[5B.1 — Tam mở liền: X có (5,4),(5,5), xét (5,6)]: creates_open_three={o3_contig}')
assert o3_contig is True
print('  -> ✓ PASS: Đánh (5,6) tạo . . X X X . ., lượt sau đi (5,3) hoặc (5,7) sẽ tạo open_four.\n')

# 2. Tam mở nhảy (có khoảng trống): X có (5,3),(5,4), xét (5,6) tạo . X X . X .
b_open3_jump = make_board(
    x_stones=[(5, 3), (5, 4)],
    o_stones=[(0, 0), (0, 2)],
)
o3_jump = creates_open_three(b_open3_jump, 5, 6, PLAYER_X)
print(f'[5B.2 — Tam mở nhảy: X có (5,3),(5,4), xét (5,6) tạo . X X . X .]: creates_open_three={o3_jump}')
assert o3_jump is True
print('  -> ✓ PASS: Đánh (5,6) để lại khe (5,5); lượt sau lấp (5,5) tạo . X X X X . (open_four).\n')

# 3. Đối chứng Negative: Tam bị O chặn sát đầu trái O X X . -> đánh (5,6) thành O X X X . .
b_blocked3 = make_board(
    x_stones=[(5, 4), (5, 5)],
    o_stones=[(5, 3), (0, 2)],
)
o3_blocked = creates_open_three(b_blocked3, 5, 6, PLAYER_X)
print(f'[5B.3 — Tam bị O chặn sát đầu trái tại (5,3), xét (5,6)]: creates_open_three={o3_blocked}')
assert o3_blocked is False
print('  -> ✓ PASS: Do (5,3) bị O chiếm, bước kế tiếp đánh (5,7) chỉ tạo broken_four chứ không tạo được open_four.\n')

# 4. Đối chứng Negative: Tam sát biên trái | X X . -> đánh (5,2) thành | X X X . .
b_edge3 = make_board(
    x_stones=[(5, 0), (5, 1)],
    o_stones=[(0, 0), (0, 2)],
)
o3_edge = creates_open_three(b_edge3, 5, 2, PLAYER_X)
print(f'[5B.4 — Tam sát biên trái (5,0),(5,1), xét (5,2)]: creates_open_three={o3_edge}')
assert o3_edge is False
print('  -> ✓ PASS: Chạm biên cột 0 nên không thể phát triển thành open_four hai đầu mở.')

=== PHẦN 5B: Thực nghiệm đối chứng Open-Three (Tam mở liền, Tam mở nhảy vs. Tam bị chặn) ===

[5B.1 — Tam mở liền: X có (5,4),(5,5), xét (5,6)]: creates_open_three=True
  -> ✓ PASS: Đánh (5,6) tạo . . X X X . ., lượt sau đi (5,3) hoặc (5,7) sẽ tạo open_four.

[5B.2 — Tam mở nhảy: X có (5,3),(5,4), xét (5,6) tạo . X X . X .]: creates_open_three=True
  -> ✓ PASS: Đánh (5,6) để lại khe (5,5); lượt sau lấp (5,5) tạo . X X X X . (open_four).

[5B.3 — Tam bị O chặn sát đầu trái tại (5,3), xét (5,6)]: creates_open_three=False
  -> ✓ PASS: Do (5,3) bị O chiếm, bước kế tiếp đánh (5,7) chỉ tạo broken_four chứ không tạo được open_four.

[5B.4 — Tam sát biên trái (5,0),(5,1), xét (5,2)]: creates_open_three=False
  -> ✓ PASS: Chạm biên cột 0 nên không thể phát triển thành open_four hai đầu mở.


### 5.3. Nhận xét kết quả thực nghiệm Phần 5

Từ kết quả kiểm chứng trực tiếp trên 9 mẫu hình đối chứng ở Phần 5A và Phần 5B (tất cả đều đạt `✓ PASS`):
- **Tính chính xác ngữ nghĩa:** Cách định nghĩa `creates_open_four` (`count_winning_threats >= 2`) và `creates_broken_four` (`count_winning_threats == 1`) phân biệt rạch ròi giữa chuỗi 4 mở hai đầu (`. X X X X .`, 2 điểm thắng tại `(5, 2)` và `(5, 7)`) với chuỗi 4 bị chặn 1 đầu bởi đối phương (`O X X X X .`), sát biên (`| X X X X .`), hoặc đứt quãng (`X X . X X`).
- **Khả năng bao quát chuỗi nhảy của `creates_open_three`:** Nhờ định nghĩa đệ quy 1 bước (kiểm tra xem nước đi có mở ra điểm tạo `open_four` mới ở lượt kế hay không), hàm tự động nhận diện được cả tam mở liền (`. . X X X . .`) lẫn tam mở có khoảng trống (`. X X . X .`) mà không cần viết biểu thức chính quy riêng.
- **Kháng nhiễu False Positive:** Khi quân đối phương chặn ở giữa (`X X O X X`) hoặc chặn sát đầu chuỗi 3 (`O X X X . .` / sát biên `| X X X . .`), các hàm đều trả về `False`, tránh việc tác tử lãng phí nước đi vào các chuỗi chết hoặc chuỗi không thể tạo đe dọa kép.

---

## Phần 6 — Giải thích quyết định nước đi

Ở chế độ `use_patterns=True`, phương thức `explain_last_action()` cung cấp bản ghi truy vết đầy đủ của nước đi vừa chọn:
- `rule_triggered`: Tên tầng luật kích hoạt trong 8 tầng (`win_now`, `block_urgent`, `attack_open_four`, `block_open_four`, `attack_broken_four`, `attack_open_three`, `block_open_three`, `positional_fallback`).
- `selected_action`: Tọa độ `(row, col)` được chọn sau khi phá hòa.
- `candidate_count`: Số lượng nước đi khớp ở tầng kích hoạt.
- `matching_actions`: Danh sách toàn bộ các tọa độ cùng thỏa mãn tầng luật đó.
- `reason`: Chuỗi giải thích bằng tiếng Việt.

Dưới đây chúng ta kiểm chứng thực nghiệm cách tác tử giải quyết xung đột giữa các tầng ưu tiên, tính đối xứng X/O, giới hạn trước đe dọa kép và so sánh trực tiếp với chế độ `use_patterns=False`.

In [16]:
def trace_pattern_scenario(board, player, expected_rule, acceptable_actions=None, title=''):
    """Chạy RuleBasedAgent(use_patterns=True) và in chi tiết truy vết explain_last_action()."""
    _agent = RuleBasedAgent(name='RuleBased-Patterns', use_patterns=True, seed=None)
    board_before = board.copy()
    action = _agent.choose_action(board_before, player)
    info = _agent.explain_last_action()
    
    board_unchanged = np.array_equal(board, board_before)
    rule_ok = info['rule_triggered'] == expected_rule
    action_ok = (acceptable_actions is None) or (action in acceptable_actions)
    status = '✓ PASS' if (rule_ok and action_ok and board_unchanged) else '✗ FAIL'
    p_sym = 'X' if player == PLAYER_X else 'O'
    
    print(f'{status} [{title}] (Lượt: {p_sym})')
    print(f'  rule_triggered  : {info["rule_triggered"]} (kỳ vọng: {expected_rule})')
    print(f'  selected_action : {info["selected_action"]}')
    print(f'  candidate_count : {info["candidate_count"]}')
    print(f'  matching_actions: {info["matching_actions"]}')
    print(f'  reason          : {info["reason"]}')
    print(f'  nodes_visited   : {_agent.stats["nodes_visited"]} | thinking_time: {_agent.stats["thinking_time"]:.6f}s | Board bất biến: {board_unchanged}')
    print()
    assert status == '✓ PASS', f'Kịch bản thất bại: {title}'
    return info

print('=== PHẦN 6A: Giải quyết xung đột giữa các tầng ưu tiên Pattern ===')
print()

# 1. Xung đột Tầng 3 vs Tầng 4: attack_open_four > block_open_four
# Cả X (hàng 5) và O (hàng 2) đều có 3 quân mở 2 đầu; đến lượt X đi -> X tự tạo open_four
b_c1 = make_board(
    x_stones=[(5, 3), (5, 4), (5, 5)],
    o_stones=[(2, 3), (2, 4), (2, 5)],
)
trace_pattern_scenario(
    b_c1, PLAYER_X, 'attack_open_four', [(5, 2), (5, 6)],
    '6A.1 — Xung đột: attack_open_four > block_open_four'
)

# 2. Xung đột Tầng 4 vs Tầng 5 & 6: block_open_four > attack_broken_four / attack_open_three
# O có 3 quân mở 2 đầu ở hàng 2: (2,3),(2,4),(2,5); X có chuỗi 3 bị chặn 1 đầu ở hàng 5: (5,0),(5,1),(5,2)
b_c2 = make_board(
    x_stones=[(5, 0), (5, 1), (5, 2)],
    o_stones=[(2, 3), (2, 4), (2, 5)],
)
trace_pattern_scenario(
    b_c2, PLAYER_X, 'block_open_four', [(2, 2), (2, 6)],
    '6A.2 — Xung đột: block_open_four > attack_broken_four'
)

# 3. Xung đột Tầng 5 vs Tầng 6: attack_broken_four > attack_open_three
# X có (5,0),(5,1),(5,2) (đánh (5,3) hoặc (5,4) tạo broken_four) và (8,4),(8,5) (tạo open_three)
b_c3 = make_board(
    x_stones=[(5, 0), (5, 1), (5, 2), (8, 4), (8, 5)],
    o_stones=[(0, 0), (0, 3), (0, 6), (0, 9), (10, 0)],
)
trace_pattern_scenario(
    b_c3, PLAYER_X, 'attack_broken_four', [(5, 3), (5, 4)],
    '6A.3 — Xung đột: attack_broken_four > attack_open_three'
)

# 4. Xung đột Tầng 6 vs Tầng 7: attack_open_three > block_open_three
# Cả X (hàng 5) và O (hàng 2) đều có 2 quân mở thoáng
b_c4 = make_board(
    x_stones=[(5, 4), (5, 5)],
    o_stones=[(2, 4), (2, 5)],
)
trace_pattern_scenario(
    b_c4, PLAYER_X, 'attack_open_three', [(5, 2), (5, 3), (5, 6), (5, 7)],
    '6A.4 — Xung đột: attack_open_three > block_open_three'
)

# 5. Xung đột Tầng 7 vs Tầng 8: block_open_three > positional_fallback
# O có 2 quân mở thoáng tại (5,4),(5,5), X chỉ có 2 quân rời rạc ở góc
b_c5 = make_board(
    x_stones=[(0, 0), (10, 10)],
    o_stones=[(5, 4), (5, 5)],
)
trace_pattern_scenario(
    b_c5, PLAYER_X, 'block_open_three', [(5, 2), (5, 3), (5, 6), (5, 7)],
    '6A.5 — Xung đột: block_open_three > positional_fallback'
)

=== PHẦN 6A: Giải quyết xung đột giữa các tầng ưu tiên Pattern ===

✓ PASS [6A.1 — Xung đột: attack_open_four > block_open_four] (Lượt: X)
  rule_triggered  : attack_open_four (kỳ vọng: attack_open_four)
  selected_action : (5, 6)
  candidate_count : 50
  matching_actions: [(5, 2), (5, 6)]
  reason          : Tấn công bốn quân mở (open four) tại (5, 6) — tìm thấy 2 nước phù hợp trong 50 ứng viên, chọn theo tie-breaking.
  nodes_visited   : 280 | thinking_time: 0.003641s | Board bất biến: True

✓ PASS [6A.2 — Xung đột: block_open_four > attack_broken_four] (Lượt: X)
  rule_triggered  : block_open_four (kỳ vọng: block_open_four)
  selected_action : (2, 6)
  candidate_count : 46
  matching_actions: [(2, 2), (2, 6)]
  reason          : Chặn đối thủ tạo bốn quân mở (open four) tại (2, 6) — tìm thấy 2 nước phù hợp trong 46 ứng viên, chọn theo tie-breaking.
  nodes_visited   : 322 | thinking_time: 0.005431s | Board bất biến: True

✓ PASS [6A.3 — Xung đột: attack_broken_four > attack_open_thre

{'rule_triggered': 'block_open_three', 'selected_action': (5, 6), 'candidate_count': 44, 'matching_actions': [(5, 2), (5, 3), (5, 6), (5, 7)], 'reason': 'Chặn đối thủ tạo ba quân mở (open three) tại (5, 6) — tìm thấy 4 nước phù hợp trong 44 ứng viên, chọn theo tie-breaking.'}

In [17]:
print('=== PHẦN 6B: Tính đối xứng góc nhìn X/O, Đe dọa kép và So sánh use_patterns ===')
print()

# 1. Kiểm chứng đối xứng góc nhìn: đến lượt O khi O có 3 quân mở (5,3),(5,4),(5,5)
b_sym_o = make_board(
    x_stones=[(0, 0), (0, 3), (0, 6), (10, 10)],
    o_stones=[(5, 3), (5, 4), (5, 5)],
)
trace_pattern_scenario(
    b_sym_o, PLAYER_O, 'attack_open_four', [(5, 2), (5, 6)],
    '6B.1 — Đối xứng góc nhìn: O kích hoạt attack_open_four tương đương X'
)

# 2. Phân tích tình huống không thể phòng thủ hoàn toàn (Đe dọa kép / Double Threat)
# O đã tạo thành công chuỗi 4 mở hai đầu tại hàng 5: (5,3),(5,4),(5,5),(5,6)
b_double_threat = make_board(
    x_stones=[(0, 0), (0, 2), (2, 0), (2, 2)],
    o_stones=[(5, 3), (5, 4), (5, 5), (5, 6)],
)
show_board(b_double_threat, '6B.2 — Thế cờ Đe dọa kép: O đã có . O O O O . tại hàng 5')
info_dt = trace_pattern_scenario(
    b_double_threat, PLAYER_X, 'block_urgent', [(5, 2), (5, 7)],
    '6B.2 — Phản ứng của X trước Đe dọa kép (. O O O O .)'
)
# Mô phỏng lượt tiếp theo của O sau khi X chặn 1 đầu
b_after_block = b_double_threat.copy()
b_after_block[info_dt['selected_action']] = PLAYER_X
remaining_o_wins = find_winning_actions(b_after_block, PLAYER_O)
print(f'  -> Sau khi X chặn tại {info_dt["selected_action"]}, O vẫn còn nước thắng ngay tại: {remaining_o_wins}')
print('  -> Giải thích: Mỗi lượt chỉ được đặt 1 quân nên không thuật toán nào chặn được đồng thời 2 điểm thắng độc lập.')
print('     Cách phòng thủ duy nhất là chặn từ tầng block_open_four / block_open_three ở các lượt trước đó!\n')

# 3. So sánh trực tiếp use_patterns=False vs use_patterns=True trên cùng thế cờ trung cuộc
# O đang có 3 quân mở 2 đầu tại (4,3),(4,4),(4,5); X có quân quanh tâm (5,5),(6,5),(0,0)
b_compare = make_board(
    x_stones=[(5, 5), (6, 5), (0, 0)],
    o_stones=[(4, 3), (4, 4), (4, 5)],
)
agent_no_pat = RuleBasedAgent(name='No-Patterns', use_patterns=False, seed=None)
agent_with_pat = RuleBasedAgent(name='With-Patterns', use_patterns=True, seed=None)

act_no = agent_no_pat.choose_action(b_compare, PLAYER_X)
info_no = agent_no_pat.explain_last_action()

act_yes = agent_with_pat.choose_action(b_compare, PLAYER_X)
info_yes = agent_with_pat.explain_last_action()

print('=== 6B.3 — Đối chứng quyết định giữa use_patterns=False và use_patterns=True ===')
print(f'  [use_patterns=False] Nước chọn: {act_no} | Luật: {info_no["rule_triggered"]} | nodes_visited: {agent_no_pat.stats["nodes_visited"]}')
print(f'    Lý do: {info_no["reason"]}')
print(f'  [use_patterns=True ] Nước chọn: {act_yes} | Luật: {info_yes["rule_triggered"]} | nodes_visited: {agent_with_pat.stats["nodes_visited"]}')
print(f'    Lý do: {info_yes["reason"]}')
assert info_no['rule_triggered'] == 'positional_fallback'
assert info_yes['rule_triggered'] == 'block_open_four' and act_yes in [(4, 2), (4, 6)]
print('  -> ✓ PASS: Bản Giai đoạn 1 bỏ sót mối đe dọa tạo 4 mở của O, trong khi bản Giai đoạn 2 chặn chính xác tại (4, 6)!')

=== PHẦN 6B: Tính đối xứng góc nhìn X/O, Đe dọa kép và So sánh use_patterns ===

✓ PASS [6B.1 — Đối xứng góc nhìn: O kích hoạt attack_open_four tương đương X] (Lượt: O)
  rule_triggered  : attack_open_four (kỳ vọng: attack_open_four)
  selected_action : (5, 6)
  candidate_count : 64
  matching_actions: [(5, 2), (5, 6)]
  reason          : Tấn công bốn quân mở (open four) tại (5, 6) — tìm thấy 2 nước phù hợp trong 64 ứng viên, chọn theo tie-breaking.
  nodes_visited   : 292 | thinking_time: 0.004345s | Board bất biến: True

=== 6B.2 — Thế cờ Đe dọa kép: O đã có . O O O O . tại hàng 5 ===
    0  1  2  3  4  5  6  7  8  9 10
 0 X  .  X  .  .  .  .  .  .  .  . 
 1 .  .  .  .  .  .  .  .  .  .  . 
 2 X  .  X  .  .  .  .  .  .  .  . 
 3 .  .  .  .  .  .  .  .  .  .  . 
 4 .  .  .  .  .  .  .  .  .  .  . 
 5 .  .  .  O  O  O  O  .  .  .  . 
 6 .  .  .  .  .  .  .  .  .  .  . 
 7 .  .  .  .  .  .  .  .  .  .  . 
 8 .  .  .  .  .  .  .  .  .  .  . 
 9 .  .  .  .  .  .  .  .  .  .  . 
10 .  .  .

### 6.3. Nhận xét và phân tích kết quả Phần 6

Từ số liệu thực tế của 8 kịch bản truy vết ở Phần 6A và Phần 6B (100% đạt `✓ PASS`):

1. **Giải quyết xung đột ưu tiên nhất quán và đối xứng:**
   - Khi cả hai bên cùng có cơ hội tạo mẫu hình ngang cấp (`attack_open_four` vs `block_open_four` ở 6A.1, hoặc `attack_open_three` vs `block_open_three` ở 6A.4), tác tử luôn ưu tiên **tấn công trước** vì bên đến lượt đi trước sẽ hoàn tất mẫu hình và ép đối thủ vào thế bị động.
   - Ngược lại, khi đối thủ đe dọa ở cấp độ cao hơn (`block_open_four` so với `attack_broken_four` ở 6A.2), tác tử lập tức ưu tiên phòng thủ chặn `open_four` tại `(2, 6)`.
   - Khi đổi vai sang `PLAYER_O` (6B.1), `explain_last_action()` trả về cùng tầng luật `attack_open_four` và chọn `(5, 6)` (gần tâm nhất theo tie-breaking), chứng minh logic hoàn toàn độc lập với ký hiệu quân cờ.

2. **Phân tích chi phí tính toán (`nodes_visited` và `thinking_time`):**
   - Ở mỗi lượt không kết thúc ở tầng 1–2, tác tử quét `2 × total_empty` node cho `win_now` và `block_urgent`, sau đó mỗi tầng pattern duyệt qua tập `candidates` (`radius=2`, từ 44–78 ô ứng viên ở các kịch bản trên). Nhờ vậy, `nodes_visited` tăng có kiểm soát (từ `226` ở tầng `block_urgent` đến `280–456` node ở các tầng pattern) và thời gian suy nghĩ `thinking_time` chỉ dao động từ **~0.6 ms đến ~115 ms** (cao nhất ở tầng `block_open_three` do kiểm tra đệ quy 1 bước), vẫn thấp hơn rất nhiều so với ngưỡng ngân sách 1.9 giây/nước.

3. **Giới hạn tự nhiên trước Đe dọa kép (Double Threat) và giá trị của Giai đoạn 2:**
   - Ở kịch bản 6B.2, khi O đã hình thành `. O O O O .` với 2 điểm thắng độc lập `[(5, 2), (5, 7)]`, tầng `block_urgent` vẫn hoạt động đúng thiết kế (chọn chặn `(5, 7)` theo tie-breaking), nhưng O tất yếu thắng ở đầu còn lại `(5, 2)` vào lượt kế.
   - Đối chứng ở kịch bản 6B.3 chỉ rõ giải pháp: khi O mới có 3 quân `. O O O .` tại hàng 4, biến thể `use_patterns=False` chỉ chọn `positional_fallback` tại `(5, 4)` (`nodes_visited = 276`), bỏ sót mối đe dọa của O; trong khi `use_patterns=True` kích hoạt `block_open_four` chặn chính xác tại `(4, 6)` (`nodes_visited = 322`) trước khi đe dọa kép kịp hình thành.

## Phần 7 — Đánh giá tác dụng thành phần (Ablation)

> **Trạng thái:** Chưa thực hiện — sẽ hoàn thiện ở Giai đoạn 3.

Nội dung dự kiến:
- So sánh Full Pattern (`use_patterns=True`) vs. No Pattern (`use_patterns=False`).
- Giữ nguyên luật thắng/chặn ở cả hai biến thể.
- Tập vị trí kiểm tra, seed và cơ chế phá hòa đồng nhất.


## Phần 8 — Thi đấu thử nghiệm

> **Trạng thái:** Chưa thực hiện — sẽ hoàn thiện ở Giai đoạn 3.

Nội dung dự kiến:
- Đấu với `RandomAgent`, đổi bên X/O, nhiều seed.
- Ghi nhận W/D/L, điểm số và `elapsed_s` trung bình.
- Dùng `evaluation/metrics.py` để tổng hợp kết quả.


## Phần 9 — Phân tích các trường hợp thất bại

> **Trạng thái:** Chưa thực hiện — sẽ hoàn thiện ở Giai đoạn 3.

Nội dung dự kiến:
- Phân tích thế cờ đối phương tạo đe dọa kép (double threat).
- Nhận diện nhầm pattern hoặc giới hạn của tầng luật hiện tại.
- Ghi nhận như hạn chế kỹ thuật, không yêu cầu giải quyết bài toán bất khả thi.


## Phần 10 — Tổng kết và hướng dẫn tái lập

> **Trạng thái:** Tổng kết Giai đoạn 1 & Giai đoạn 2 bên dưới; số liệu thực nghiệm Ablation và thi đấu nhiều seed sẽ cập nhật đầy đủ ở Giai đoạn 3.

### Tổng kết Giai đoạn 1 & Giai đoạn 2

**Câu trả lời tạm thời (trước khi chạy thực nghiệm thi đấu quy mô lớn ở Giai đoạn 3):**  
- **Giai đoạn 1 (`use_patterns=False`):** Luật thắng ngay (`win_now`) và chặn thua (`block_urgent`) hoạt động chính xác tuyệt đối ở cả 4 hướng và 2 phía (X/O).
- **Giai đoạn 2 (`use_patterns=True`):** Bổ sung 4 hàm nhận diện mẫu hình (`count_winning_threats`, `creates_open_four`, `creates_broken_four`, `creates_open_three`) và bảng ưu tiên 8 tầng giúp tác tử chủ động tạo đe dọa (`attack_open_four`, `attack_broken_four`, `attack_open_three`) cũng như chặn sớm các thế cờ nguy hiểm (`block_open_four`, `block_open_three`) trước khi đối thủ kịp hình thành đe dọa kép.

**Hạn chế kỹ thuật đã biết:**
- Khi đối thủ đã tạo thành công đe dọa kép (ví dụ `. O O O O .` hoặc giao điểm đa đe dọa thắng ngay), tác tử chỉ chặn được 1 điểm trong lượt đi của mình.
- Các tầng pattern (tầng 3–7) chỉ quét trong tập ứng viên `candidate_actions(radius=2)` để tối ưu tốc độ (~1–4 ms/nước), nên có thể bỏ qua một số cấu trúc nhảy rất xa đám quân hiện tại.

**Cấu hình đề xuất:**
- Baseline chiến thuật (Giai đoạn 1): `RuleBasedAgent(candidate_radius=2, use_patterns=False, seed=None)`
- Đầy đủ Pattern (Giai đoạn 2): `RuleBasedAgent(candidate_radius=2, use_patterns=True, seed=None)`

**Thứ tự chạy lại notebook:**  
Restart Kernel → Run All. Không phụ thuộc vào trạng thái biến từ cell trước.
